# Bayesian Filtering

## Thinking About Belief
Baye's Law is a mechanism for updating our prior beliefs based on the presentation of new evidence. Bayesian filtering is a family of algorithms for performing these updates of belief over time as we receive new information.

In Bayesian language, we describe belief as a probability distribution rather than trying to guess some single, discrete value. Instead of thinking things like

> I think we are going X miles an hour

We describe our belief with phrases like

> I think we are going between X and Y miles per hour

This second way of framing belief aligns with our intuition about probability distributions. Probability distributions tell us about a range of potential outcomes rather than specifying a single one. 

## Notation

Since we are talking about probability, we will distinguish random variables from their realized, concrete values. 

$$
\begin{align}
X_t &:= \text{A random variable representing our unkown position at time } t \\
x_t &:= \text{A realized value of our random variable at time } t
\end{align}
$$

## Bayes Law

I'll put this here now, but we aren't going to talk about how it relates to bayesian filtering. That will be made apparent as we step through the algorithm. 

$$
P(H | E) = \frac{P( E | H ) P(H)}{P(E)}
$$

## The Algorithm

### Step 1: Establishing Prior Belief
Let's say we are concerned with ascertaining our position over time. At time 0 we might have some belief about where we are, expressed as a probability distribution:

$$
P(X_0)
$$

This initial belief could be an educated guess, or it might come from some measurement. 

### Step 2: Receiving New Evidence
At this point, we get a measurement from a sensor, perhaps GPS, telling us that our location is $z_0$. Note that the notation is lower case, and therefore is not a random variable but an actual measured value. 

### Step 3: Forming Likelihood Function

Likelihood answers the question 

> "Assuming the true position was $X_t = x_t$, how likely is the measurement $z_t$ I recieved?" 

In order to define likelihood mathematically, we consider that our measurement was generated by some imperfect process. For example, we might model the process of generating GPS measurement as a function of our true position plus some random error $V$. We might know, because the manufacturere told us, that the error follows a normal distribution with known variance $\sigma^2$ and mean $\mu = 0$:

$$
\begin{align}
Z_t = X_t + V, \\
V \sim \mathcal{N}(\mu, \sigma^2)
\end{align}
$$

Now, we will let $X_t = x_t$ be one such candidate value of the unknown true state. Then, what can we say about the conditional distribution of $Z_t$? Well, since adding a fixed value $x_t$ to a zero-mean normal distribution just shifts the mean of the distribution by $x_t$, the conditional distribution of the measurement is:

$$
Z_t \mid X_t = x_t \sim \mathcal{N}(x_t, \sigma^2)
$$

We know that the probability density function of a normal distribution of a random variable $Z$ with mean $\mu$ is defined as:

$$
p(z_t; \mu_z, \sigma) = \frac{1}{\sqrt{2 \pi \sigma^2}} \exp \bigg(-\frac{(z_t - \mu)^2}{2 \sigma^2} \bigg)
$$

To construct the likelihood function at time $t=0$

1. We fix the value of $z_t$ to $z_0$, the measurement we received from the GPS.
2. We already know the variance $\sigma^2$ is just the error distribution of the GPS (we got its value from the sensor's manufacturer).
3. From our measurement model, we know that if the true state is $x_0$, then the measurement distribution has mean $\mu = x_0$.

So, our likelihood function (again, only a function of $x_0$) becomes

$$
p(z_0 \mid x_0) = \frac{1}{\sqrt{2 \pi \sigma^2}} \exp \bigg(-\frac{(z_0 - x_0)^2}{2 \sigma^2} \bigg)
$$

And it answers the question

> "how likely is it that the GPS reported $z_0$ given different possible true states $x_0$. 

Another way to think about is that ***"the likelihood assigns a score to each candidate true state according to how well that candidate explains the observed measurement"***.

#### Visualizing Likelihood

Likelihood is one of the more subtle concepts in Bayesian inference, so it's worth looking at it from a few different perspectives. The interactive plot below may also help build intuition.

**Definition:** The likelihood is the probability density of observing the measurement we received, assuming a particular hypothesis about the true state is correct.

**Interpretation:** Another way to think about it is, "How surprised would I be to observe this measurement if this hypothesis were true?" A higher likelihood means the measurement is less surprising under that hypothesis.

Once the measurement has been observed, however, it is fixed. The only quantity that varies is the candidate hypothesis about the true state. We can therefore view the likelihood as a scoring function that compares how consistent different candidate states are with the observed measurement. Higher likelihood values indicate that a candidate state is more consistent with the observed measurement than one with a lower likelihood.


In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets

sns.set_theme(context='notebook', style='darkgrid')
from functools import partial


def normal_pdf(x, mean=0.0, std=1.0):
    variance = std ** 2
    denominator = np.sqrt(2 * np.pi * variance)
    
    numerator = np.exp(-((x - mean) ** 2) / (2 * variance))
    return numerator / denominator


start, stop, step = 0, 10, 0.01
@widgets.interact(z0=widgets.FloatSlider(value=stop/2, min=start, max=stop, layout=widgets.Layout(width='60%')), 
                  x0=widgets.FloatSlider(value=stop/2, min=start, max=stop, step=step, layout=widgets.Layout(width='60%')))
def likelihood_widget(z0, x0):

    likelihood = partial(normal_pdf, z0)
    
    x = np.arange(start, stop, step)
    likelihood_curve = likelihood(x)

    score = likelihood(x0)
    
    fig, ax = plt.subplots(figsize=(8, 5))

    # 2. Define text box styling properties
    box_properties = dict(
        boxstyle="round,pad=0.5",  # Shape and padding
        facecolor="wheat",  # Background color
        edgecolor="orange",  # Border color
        alpha=0.8,  # Transparency
    )
    
    # 3. Add the text box to the plot
    ax.text(
        x=start,  # X-coordinate (matches data units)
        y=max(likelihood_curve) / 2,  # Y-coordinate (matches data units)
        s=f"measurement: {z0} \nlikelihood score: {score:.2f}",  # Text content
        fontsize=11,
        color="black",
        bbox=box_properties,  # Activates the text box
    )
    
    sns.lineplot(x=x, y=likelihood_curve, ax=ax, label="likelihood curve")
    ax.vlines(x=x0, ymin=0, ymax=score, color='red', linewidth=1)
    sns.scatterplot(x=[x0], y=score, ax=ax, c='red', label="likelihood")
    plt.show()



interactive(children=(FloatSlider(value=5.0, description='z0', layout=Layout(width='60%'), max=10.0), FloatSli…

### Step 4: Updating Our Belief

Recall that we said that our belief about the true state at time $t$ was described using $p(X_t)$. This is not a statement of belief that a single value is the right one, but rather it describes our belief about a range of potential values. What can we say about the following expression?

$$
f(X_0) = p(X_0)p(z_0 \mid X_0)
$$

If $p(X_0)$ is our prior belief about our true state, then this function scales that belief by the likelihood of the measurement we received. In other words, for each potential true position $X_0 = x_0$, $$p(x_0)p(z_0 \mid x_0)$$ gives us a scaled version of our original belief about the value probability of $x_0$ being correct. 

In the language of Bayesian statistics, we are updating our original belief, or hypothesis, by the likelihood of our evidence, the recieved GPS measurement. 


In [5]:
from functools import Placeholder

def update_belief(x, prior, likelihood):
    return prior(x) * likelihood(x)


start, stop, step = 0, 10, 0.01
@widgets.interact(z0=widgets.FloatSlider(value=stop/2, min=start, max=stop, layout=widgets.Layout(width='60%')), 
                  x0=widgets.FloatSlider(value=stop/2, min=start, max=stop, step=step, layout=widgets.Layout(width='60%')),
                  candidate_state=widgets.FloatSlider(value=stop/2, min=start, max=stop, step=step, layout=widgets.Layout(width='60%')))
def update_widget(z0, x0, candidate_state):

    x = np.arange(start, stop, step)
    
    likelihood = partial(normal_pdf, z0)
    likelihood_curve = likelihood(x)
    prior_curve = normal_pdf(x, x0)

    update_curve = update_belief(x, partial(normal_pdf, Placeholder, x0), likelihood)
    update_score = update_belief(candidate_state, partial(normal_pdf, Placeholder, x0), likelihood)
    
    fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=False)
    
    sns.lineplot(x=x, y=prior_curve, ax=ax[0], label="Prior Belief $p(X_0)$")
    sns.lineplot(x=x, y=likelihood_curve, ax=ax[0], label="Likelihood $p(Z_0 \\mid X_0)$")
    sns.lineplot(x=x, y=update_curve, ax=ax[0], label="Updated Belief")

    sns.lineplot(x=x, y=update_curve, ax=ax[1])
    ax[1].vlines(x=candidate_state, ymin=0, ymax=update_score, color='red', linewidth=1)
    sns.scatterplot(x=[candidate_state], y=update_score, ax=ax[1], c='red', label="likelihood")

        # 2. Define text box styling properties
    box_properties = dict(
        boxstyle="round,pad=0.5",  # Shape and padding
        facecolor="wheat",  # Background color
        edgecolor="orange",  # Border color
        alpha=0.8,  # Transparency
    )
    
    # 3. Add the text box to the plot
    ax[1].text(
        x=start,  # X-coordinate (matches data units)
        y=max(update_curve) / 2,  # Y-coordinate (matches data units)
        s=f"Belief score for candidate state: {update_score:.2f}",  # Text content
        fontsize=11,
        color="black",
        bbox=box_properties,  # Activates the text box
    )
    

    ax[1].get_legend().remove()
    plt.show()

interactive(children=(FloatSlider(value=5.0, description='z0', layout=Layout(width='60%'), max=10.0), FloatSli…